In [ ]:
#Install Dependencies

In [ ]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q -U huggingface_hub

In [ ]:
#Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#Load Dataset

In [ ]:
import os
import numpy as np
import pandas as pd
import faiss

RAG_DIR = "/content/drive/MyDrive/Medical-RAG"

INDEX_PATH = os.path.join(
    RAG_DIR,
    "medical_rag.index"
)

DOCUMENTS_PATH = os.path.join(
    RAG_DIR,
    "documents.pkl"
)

EMBEDDINGS_PATH = os.path.join(
    RAG_DIR,
    "embeddings.npy"
)

print("RAG directory:", RAG_DIR)

In [ ]:
#load model

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

In [ ]:
#rag

In [ ]:
index = faiss.read_index(INDEX_PATH)

documents = pd.read_pickle(
    DOCUMENTS_PATH
)

embeddings = np.load(
    EMBEDDINGS_PATH
)

print("FAISS index:", index.ntotal)
print("Documents:", len(documents))
print("Embeddings:", embeddings.shape)

In [ ]:
print(documents.head())
print("\nColumns:")
print(documents.columns.tolist())

In [ ]:
def retrieve(query, top_k=5, max_distance=0.65):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = documents.iloc[indices[0]].copy()

    results["distance"] = distances[0]

    # Remove weak matches
    results = results[
        results["distance"] <= max_distance
    ]

    return results

In [ ]:
#hugginface

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import InferenceClient

hf_client = InferenceClient(
    provider="hf-inference"
)

print("Hugging Face client initialized.")

In [ ]:
#rag

In [ ]:
def ask_huggingface(question, k=5):

    results = retrieve(question, k)

    # No sufficiently relevant research
    if len(results) == 0:
        return (
            "The retrieved research does not contain enough information "
            "to answer this.",
            results
        )

    context = "\n\n".join(
        f"Abstract ID: {row.abstract_id}\n"
        f"{row.abstract_text}"
        for _, row in results.iterrows()
    )

    prompt = f"""
You are an AI biomedical research assistant.

Answer the question ONLY using the supplied research abstracts.

Rules:
- Give a concise answer in 1-2 complete sentences.
- Do not speculate.
- Do not add information that is not supported by the abstracts.
- Do not provide citations or source IDs.
- If the abstracts do not contain enough information, say:
  "The retrieved research does not contain enough information to answer this."

RESEARCH:

{context}

QUESTION:

{question}

ANSWER:
"""

    response = hf_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=500
    )

    answer = response.choices[0].message.content.strip()

    # Generate source list in Python
    source_ids = [
        str(row.abstract_id)
        for _, row in results.iterrows()
    ]

    sources = "\n".join(
        f"- Abstract ID: {source_id}"
        for source_id in source_ids
    )

    final_answer = (
        f"{answer}\n\n"
        f"Sources:\n"
        f"{sources}"
    )

    return final_answer, results

In [ ]:
#eval

In [ ]:
test_questions = [
    "What does ACE2 convert angiotensin II into?",

    "What is the effect of ACE inhibitors on blood pressure in hypertensive patients?",

    "How was HIV-1 RNA measured in patients with residual viremia?",

    "How was HCMV DNA detected in infected children?",

    "How was HSV viral load measured during herpes labialis?",

    "How was hepatitis C viral load monitored during interferon treatment?",

    "What was the effectiveness of the Pfizer-BioNTech COVID-19 vaccine against the Omicron variant?"
]

In [ ]:
for i, question in enumerate(test_questions, 1):

    answer, results = ask_huggingface(question)

    print("\n" + "=" * 100)
    print(f"TEST {i}")
    print("=" * 100)

    print(f"\nQUESTION:\n{question}")

    print(f"\nANSWER:\n{answer}")

    print(f"\nRETRIEVED SOURCES: {len(results)}")

    for j, (_, row) in enumerate(results.iterrows(), 1):

        print(
            f"  [{j}] Abstract ID: {row.abstract_id} "
            f"| Distance: {row.distance:.4f}"
        )

In [ ]:
#RETRIEVAL INSPECTION

In [ ]:
question = "What does ACE2 convert angiotensin II into?"

results = retrieve(question)

for i, (_, row) in enumerate(results.iterrows(), 1):

    print("=" * 100)
    print(f"SOURCE {i}")
    print("=" * 100)

    print("Abstract ID:", row.abstract_id)
    print("Distance:", round(row.distance, 4))
    print()
    print(row.abstract_text)
    print()